# RQ3 Validity

To what extent are the boundary inputs found by the search valid inputs of the task? The notebook scores the crossing inputs with the descriptive proxies of results/validity_MAIN.csv: image distance against the codec reconstruction of the seed, text distance, the visible text view and question preservation. The proxies are descriptive and do not establish validity.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from mmb.analysis import stats as st, paired as pr, export as ex

RESULTS = Path("../results") if Path("../results").exists() else Path("results")
FIG = RESULTS / "figures"; FIG.mkdir(parents=True, exist_ok=True)
GRID = ["qwen", "internvl", "ministral"]      # the three SUTs of the study (balanced grid)
REF = "a0_directed"                            # MMBA, the reference of every paired test
N_BOOT = 2000                                  # the paper uses 10000; raise for the final numbers
df = st.load_results([RESULTS / "MAIN.csv"])
scopes = ex.scopes(df, GRID)                   # {(sut, task): frame}; ("All", "All") pools the grid
grid_df = scopes[("All", "All")]
print(len(df), "runs;", "arms:", sorted(df["arm"].unique()))
from mmb.analysis import validity_cost as vc
vdf = pd.read_csv(RESULTS / "validity_MAIN.csv")
print(vdf["view"].value_counts().to_dict())

## Median proxies per arm

Medians of CLIP, DINOv2, LPIPS, SSIM, MiniLM cosine and word edits at the first crossing input.

In [ ]:
v1 = vdf[vdf["view"] == "first"] if "view" in vdf else vdf
v1.groupby("arm")[[c for c in ("img_clip_cos", "img_dino_cos", "img_lpips", "img_ssim", "txt_minilm_cos", "txt_word_edit") if c in v1]].median().round(3)

## Share of crossings under the a-priori thresholds

Thresholds LPIPS below 0.2 and text cosine distance below 0.1 were fixed before the numbers existed. The conservative share counts every run of the arm in the denominator. Two views are defined in advance: the first crossing input and the search's own minimal-edit crossing.

In [ ]:
for view in ("first", "min"):
    print(view)
    display(vc.validity_cost(vdf, df, GRID, view=view))

## Sensitivity to the thresholds

The same shares for neighbouring thresholds, to show how much the reading depends on the fixed values.

In [ ]:
vc.sensitivity(vdf, df, GRID, "first")

## Visible text view

Both questions are normalised (NFKC, zero-width characters removed, homoglyphs mapped to ASCII) before comparing. Columns visid_* and visone_* in the table above give the share of visibly identical crossings and the share within one character edit per word, per arm and view.

In [ ]:
cols = [c for c in vc.validity_cost(vdf, df, GRID, view="min").columns if c.startswith(("med_word_edit", "obj_", "med_pair"))]
vc.validity_cost(vdf, df, GRID, view="min")[["task", "arm"] + cols] if cols else print("no smallest-crossing view yet")